<a href="https://colab.research.google.com/github/shivayya03/from-scratch-movie-recommender/blob/main/Movie_Recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [60]:
import math
from collections import Counter
import numpy as np

TF-IDF vectors

In [61]:
# Create TF-IDF vectors
def create_tfidf_vectors(movie_tags):

    vocab = sorted({word for tag in movie_tags for word in tag.split()})
    doc_sets = [set(tag.split()) for tag in movie_tags]
    N = len(movie_tags)

    df_counts = {
        term: sum(1 for doc_set in doc_sets if term in doc_set)
        for term in vocab
    }
    idf = {
        term: math.log(N / df_counts[term]) + 1
        for term in vocab
    }

    movie_vectors = []

    for tag in movie_tags:

        words = tag.split()
        length = len(words)
        word_counts = Counter(words)

        row = [
            round((word_counts[term] / length) * idf[term], 2)
            for term in vocab
        ]

        movie_vectors.append(row)

    return movie_vectors


Cosine Similarity

In [62]:
# Cosine similarity formula
def cosine_similarity(vec1, vec2):

    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    magnitude1 = math.sqrt(sum(num ** 2 for num in vec1))
    magnitude2 = math.sqrt(sum(num ** 2 for num in vec2))

    if magnitude1 == 0 or magnitude2 == 0:
        return 0.0

    return dot_product / (magnitude1 * magnitude2)

In [63]:
# Create similarity matrix
def create_similarity_matrix(movie_vectors):
    movie_vectors_np = np.array(movie_vectors)

    # Calculate magnitudes for all vectors
    magnitudes = np.linalg.norm(movie_vectors_np, axis=1)

    # Calculate dot products between all pairs of vectors
    dot_products = np.dot(movie_vectors_np, movie_vectors_np.T)

    # Handle zero magnitudes to avoid division by zero
    magnitude_product = np.outer(magnitudes, magnitudes)

    # Perform division, setting similarity to 0 where magnitude_product is 0
    similarity_matrix_np = np.divide(dot_products, magnitude_product,
                                     out=np.zeros_like(dot_products),
                                     where=magnitude_product != 0)

    # Round to 2 decimal places as in the original code
    similarity_matrix_np = np.round(similarity_matrix_np, 2)

    return similarity_matrix_np.tolist()

data preprocessing

In [64]:
import pandas as pd
import ast
from nltk.stem.porter import PorterStemmer

In [65]:
ps = PorterStemmer()

# Convert genres and keywords into list
def get_names(text):

    L = []

    try:

        for i in ast.literal_eval(text):

            L.append(i['name'])

    except:

        pass

    return L

In [66]:
# Get top 3 cast members
def get_top3(text):

    L = []

    counter = 0

    try:

        for i in ast.literal_eval(text):

            if counter != 3:

                L.append(i['name'])

                counter += 1

            else:

                break

    except:

        pass

    return L

In [67]:
# Get director name
def get_director(text):

    L = []

    try:

        for i in ast.literal_eval(text):

            if i['job'] == 'Director':

                L.append(i['name'])

    except:

        pass

    return L

In [68]:
# Apply stemming
def stem(text):

    return " ".join(
        [ps.stem(word) for word in text.split()]
    )

In [69]:
# Main preprocessing function
def load_and_preprocess(sample_size=None):

    movies = pd.read_csv("tmdb_5000_movies.csv", engine='python', on_bad_lines='warn')

    credits = pd.read_csv("tmdb_5000_credits.csv", engine='python', on_bad_lines='warn')

    # Merge datasets
    df = movies.merge(credits, on='title')

    # Select useful columns and create a copy to avoid SettingWithCopyWarning
    df = df[
        [
            'movie_id',
            'title',
            'overview',
            'genres',
            'keywords',
            'cast',
            'crew'
        ]
    ].copy()

    # Remove null values
    df.dropna(inplace=True)

    # Use a smaller sample for faster performance when requested
    if sample_size is not None:
        sample_size = min(sample_size, len(df))
        df = df.sample(n=sample_size, random_state=42).reset_index(drop=True)

    # Convert JSON strings
    df['genres'] = df['genres'].apply(get_names)

    df['keywords'] = df['keywords'].apply(get_names)

    df['cast'] = df['cast'].apply(get_top3)

    df['crew'] = df['crew'].apply(get_director)

    # Split overview words
    df['overview'] = df['overview'].apply(
        lambda x: x.split()
    )

    # Remove spaces
    for col in ['genres', 'keywords', 'cast', 'crew']:

        df[col] = df[col].apply(

            lambda x: [
                i.replace(" ", "")
                for i in x
            ]
        )

    # Create tags column
    df['tags'] = (
        df['overview']
        + df['genres']
        + df['keywords']
        + df['cast']
        + df['crew']
    )

    # Final dataframe
    # Create a new DataFrame for new_df to avoid SettingWithCopyWarning
    new_df = df[['movie_id', 'title', 'tags']].copy()

    # Convert list to string
    new_df['tags'] = new_df['tags'].apply(
        lambda x: " ".join(x)
    )

    # Lowercase
    new_df['tags'] = new_df['tags'].apply(
        lambda x: x.lower()
    )

    # Stemming
    new_df['tags'] = new_df['tags'].apply(stem)

    return new_df.reset_index(drop=True)

main.py

In [70]:
# Load processed data (use a smaller sample for faster testing)
movies = load_and_preprocess(sample_size=500)

### Movie Recommendation System

In [76]:
# Get movie names and fast lookup map
movie_names = movies['title'].tolist()
movie_indices = {name: idx for idx, name in enumerate(movie_names)}

# Get movie tags
movie_tags = movies['tags'].tolist()

# Create TF-IDF vectors
movie_vectors = create_tfidf_vectors(movie_tags)

# Create similarity matrix
similarity_matrix = create_similarity_matrix(movie_vectors)

while True:
    # User input
    movie_name = input("Enter Movie Name: ")

    # Check movie
    if movie_name not in movie_indices:
        print("Movie Not Found. Please try again.")
        continue
    else:
        break

# Get movie index
movie_index = movie_indices[movie_name]

# Get similarity scores
distances = similarity_matrix[movie_index]

# Add indexes
movie_list = list(enumerate(distances))

# Sort movies
movie_list = sorted(
    movie_list,
    key=lambda x: x[1],
    reverse=True
)

print(f"\nTop 5 Movies Similar to '{movie_name}':\n")

# Print top 5 movies
for movie in movie_list[1:6]:

    movie_title = movies.iloc[movie[0]].title

    similarity_score = movie[1]

    print(movie_title, "->", similarity_score*100,"%")

Enter Movie Name: My Sister's Keeper

Top 5 Movies Similar to 'My Sister's Keeper':

Hard to Be a God -> 15.0 %
The Walking Deceased -> 12.0 %
The Work and the Glory II: American Zion -> 12.0 %
Stolen Summer -> 11.0 %
Black November -> 11.0 %


### Available Movie Titles

Please select a movie from the list below to get recommendations:

In [73]:
for name in movie_names:
    print(name)

The Tree of Life
The Secret Life of Bees
Rambo III
Solaris
Harry Potter and the Half-Blood Prince
My Sister's Keeper
The Grey
The Smurfs
200 Cigarettes
Ghost Dog: The Way of the Samurai
Piranha 3D
UnDivided
Like Mike
To Rome with Love
Kill Bill: Vol. 1
Blood Diamond
Boogie Nights
Mrs. Doubtfire
The Color Purple
Exam
Pandaemonium
Patton
Alpha and Omega
Knight and Day
Northfork
Death Race 2000
Snow Flower and the Secret Fan
(500) Days of Summer
Gone with the Wind
The Princess Diaries
The Conjuring
Beer League
10 Cloverfield Lane
Opal Dream
Blue Jasmine
Old Dogs
The Golden Compass
Paint Your Wagon
Lincoln
The Young Messiah
X-Men: The Last Stand
Woman in Gold
L.I.E.
Iron Man 2
The Front Page
Penguins of Madagascar
Underworld: Rise of the Lycans
Catch a Fire
Don Jon
The Midnight Meat Train
Highway
The Fifth Estate
Barney's Great Adventure
The Last Witch Hunter
Road Hard
Reign of Fire
Dude Where's My Dog?
Good Luck Chuck
8: The Mormon Proposition
Mongol: The Rise of Genghis Khan
Hachi: A Dog